In [1]:
%reload_ext autoreload
%autoreload 2
import pandas as pd
import gseapy as gp
import matplotlib.pyplot as plt
import os
import numpy as np

In [2]:
## define path
basedir = "/Users/kyokokurihara/iLab/itolab_backup/backup-latest/Lab/projects/2507blastx"

# output directory
sub_out = "hepatovirus_260113_MSigDB_Hallmark_2020_stat/"
out_folder_path = f"{basedir}/output/250909_4474_samples/final_test/"
out_path = out_folder_path + sub_out
plot_path = out_path + "dotplot_hepatovirus/"

# input file
input_folder_path = f"{basedir}/data/251215_rna_seq/To_kurihara_hepatovirus/output/"

# make directories
if not(os.path.exists(out_folder_path)):
    os.mkdir(out_folder_path)
if not(os.path.exists(out_path)):
    os.mkdir(out_path)
if not(os.path.exists(plot_path)):
    os.mkdir(plot_path)

print("saving files to:", plot_path)

saving files to: /Users/kyokokurihara/iLab/itolab_backup/backup-latest/Lab/projects/2507blastx/output/250909_4474_samples/final_test/hepatovirus_260113_MSigDB_Hallmark_2020_stat/dotplot_hepatovirus/


In [3]:
def plot_prerank(df, outdir_path, metric, gene_col, gene_sets):
    """
    prerank tool plot.

    metric: ranking metric
    gene_col: gene name col for enrichr
    """
    # cols: gene_name_hs, gene_name_gallus, baseMean, log2FoldChange, lfcSE, stat, pvalue, padj    
    df2 = df.copy()
    
    # drop NaN
    df2 = df2.dropna(subset=[gene_col, metric])
    df2[gene_col] = df2[gene_col].astype(str)
    # df2[gene_col] = df2[gene_col].astype(str).str.upper()
    
    # for duplicate genes, keep the one with the larger absolute value 
    # (for ties: 1. pvalue, 2. gene name)
    df2["_abs"] = df2[metric].astype(float).abs()
    df2["_p"] = df2["pvalue"].astype(float).fillna(1.0)

    df2 = (df2.sort_values([gene_col, "_abs", "_p", gene_col],
                           ascending=[True, False, True, True],
                           kind="mergesort").drop_duplicates(subset=[gene_col], keep="first"))

    # ranking (for ties: 1. pvalue, 2. gene name)
    df2 = df2.sort_values([metric, "_p", gene_col],
                          ascending=[False, True, True],
                          kind="mergesort")
    rnk = df2[[gene_col, metric]]
    print("\nrnk\n", rnk.head(3))

    # prerank
    pre_res = gp.prerank(
        rnk=rnk,
        gene_sets=gene_sets,
        outdir=f"{outdir_path}_{gene_sets}_{metric}_{gene_col}",
        seed=0,
        threads=4,
        verbose=True
    )

    # plot top 5 pathways
    terms = pre_res.res2d["Term"]
    pre_res.plot(
        terms=terms[:5], 
        legend_kws={'loc': (1.15, 0)},
        ofname=f"{outdir_path}_{gene_sets}_{metric}_{gene_col}_top5_enriched.png"
    )

In [4]:
# check Enricher library
names = gp.get_library_name()
# print("all library:", names)
print("KEGG library:", [k for k in names if k.startswith("KEGG_")])
print("MSigDB library:", [m for m in names if m.startswith("MSigDB_")])
print("Reactome library:", [r for r in names if r.startswith("Reactome")])
print("GO library:", [g for g in names if g.startswith("GO_Biological")])

KEGG library: ['KEGG_2013', 'KEGG_2015', 'KEGG_2016', 'KEGG_2019_Human', 'KEGG_2019_Mouse', 'KEGG_2021_Human', 'KEGG_2026']
MSigDB library: ['MSigDB_Computational', 'MSigDB_Hallmark_2020', 'MSigDB_Oncogenic_Signatures']
Reactome library: ['Reactome_2022', 'Reactome_Pathways_2024']
GO library: ['GO_Biological_Process_2021', 'GO_Biological_Process_2023', 'GO_Biological_Process_2025']


In [5]:
configs = {
    "metric": "stat",
    "gene_col": "gene_name_hs",
    "gene_sets": "MSigDB_Hallmark_2020"
}

for dirpath, dirnames, filenames in os.walk(input_folder_path):
    for fname in filenames:
        if not(fname.startswith("result")):
            continue

        # read DESeq2 result table
        path = os.path.join(dirpath, fname)
        project = pd.read_table(path)

        # plot prerank
        proname = fname.split("_")[1]
        print("\nprocessing...", proname)
        plot_prerank(project, out_path + proname, configs["metric"], configs["gene_col"], configs["gene_sets"])

2026-04-03 00:05:20,557 [WARNING] Duplicated values found in preranked stats: 1.38% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-04-03 00:05:20,557 [INFO] Parsing data files for GSEA.............................
2026-04-03 00:05:20,566 [INFO] Enrichr library gene sets already downloaded in: /Users/kyokokurihara/.cache/gseapy, use local file
2026-04-03 00:05:20,571 [INFO] 0000 gene_sets have been filtered out when max_size=500 and min_size=15
2026-04-03 00:05:20,571 [INFO] 0050 gene_sets used for further statistical testing.....
2026-04-03 00:05:20,572 [INFO] Start to run GSEA...Might take a while..................



processing... PRJNA774885

rnk
       gene_name_hs      stat
19233          PI3  7.624489
14275       STOML1  6.930276
9447         SPON2  5.743817


2026-04-03 00:05:23,816 [INFO] Congratulations. GSEApy runs successfully................

2026-04-03 00:05:24,002 [WARNING] Duplicated values found in preranked stats: 1.49% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-04-03 00:05:24,003 [INFO] Parsing data files for GSEA.............................
2026-04-03 00:05:24,011 [INFO] Enrichr library gene sets already downloaded in: /Users/kyokokurihara/.cache/gseapy, use local file
2026-04-03 00:05:24,016 [INFO] 0000 gene_sets have been filtered out when max_size=500 and min_size=15
2026-04-03 00:05:24,016 [INFO] 0050 gene_sets used for further statistical testing.....
2026-04-03 00:05:24,017 [INFO] Start to run GSEA...Might take a while..................



processing... PRJEB63475

rnk
       gene_name_hs       stat
16764         XPO6  12.645954
3728        PRKAG2  12.602400
4515          BRD9  11.994078


2026-04-03 00:05:28,285 [INFO] Congratulations. GSEApy runs successfully................



In [6]:
# obtain terms with FDR < 0.05 and top k
terms = set()

for dirpath, dirnames, filenames in os.walk(out_path):
    for fname in filenames:
        if fname != "gseapy.gene_set.prerank.report.csv":
            continue

        # read DESeq2 result table
        path = os.path.join(dirpath, fname)
        res = pd.read_csv(path)
        project = path.split("/")[-2].split("_")[0]

        # extract top 10 and worst 10
        top10   = res.sort_values("NES", ascending=False).head(10).copy()
        worst10 = res.sort_values("NES", ascending=True).head(10).copy()
        df = pd.concat([top10, worst10], ignore_index=True).sort_values("NES", ascending=True).reset_index(drop=True)

        # add significant terms
        for t in df.loc[df["FDR q-val"] < 0.05, "Term"].to_list():
            terms.add(t)

# show terms
print("top 10 and FDR<0.05:", len(terms)) 

# convert to list
terms = sorted(list(terms))

top 10 and FDR<0.05: 14


In [7]:
# calc mean and set order
first_file = True

for dirpath, dirnames, filenames in os.walk(out_path):
    for fname in filenames:
        if fname != "gseapy.gene_set.prerank.report.csv":
            continue

        # read DESeq2 result table
        path = os.path.join(dirpath, fname)
        res = pd.read_csv(path)
        project = path.split("/")[-2].split("_")[0]

        # extract rows with terms of interest
        df_term = res.loc[res["Term"].isin(terms), ["Term", "NES"]].copy()
        df_term.columns = df_term.columns.map(lambda x: f"{x}_{project}" if x != "Term" else x)
        print(f"Project: {project} size: {df_term.shape}")

        # merge tables across projects
        if first_file:
            df_all = df_term
            first_file = False
        else:
            df_all = df_all.merge(df_term, on="Term", how="left")

# size
print("df_all size:", df_all.shape)

# sort by mean NES
df_all["mean_NES"] = df_all[["NES_PRJNA774885", "NES_PRJEB63475"]].mean(axis=1)
df_sorted = df_all.sort_values("mean_NES", ascending=True)

# store terms order
terms_order = df_sorted["Term"].to_list()

# show
print(terms_order)

Project: PRJNA774885 size: (14, 2)
Project: PRJEB63475 size: (14, 2)
df_all size: (14, 3)
['Oxidative Phosphorylation', 'Fatty Acid Metabolism', 'Myc Targets V1', 'Adipogenesis', 'Pperoxisome', 'Reactive Oxygen Species Pathway', 'mTORC1 Signaling', 'Xenobiotic Metabolism', 'Protein Secretion', 'Coagulation', 'IL-2/STAT5 Signaling', 'IL-6/JAK/STAT3 Signaling', 'Interferon Gamma Response', 'Interferon Alpha Response']


In [8]:
# creates table for R
first_file=True
dfs = []

for dirpath, dirnames, filenames in os.walk(out_path):
    for fname in filenames:
        if fname != "gseapy.gene_set.prerank.report.csv":
            continue

        # read DESeq2 result table
        path = os.path.join(dirpath, fname)
        res = pd.read_csv(path)
        project = path.split("/")[-2].split("_")[0]

        # add -log10(FDR) (clipped)
        res["mlog10_fdr"] = -np.log10(res["FDR q-val"].clip(lower=1e-10))

        # extract rows with terms of interest
        df_term = res.loc[res["Term"].isin(terms), ["Term", "NES", "FDR q-val", "mlog10_fdr"]].copy()
        df_term["Project"] = project
        df_term_sorted = df_term.sort_values(
            "Term",
            key=lambda s: pd.Categorical(s, categories=terms_order, ordered=True)
        )

        dfs.append(df_term_sorted)

# merge
df_cat = pd.concat(dfs, ignore_index=True)
# save
df_cat.to_csv(plot_path+"NES_termshared_top10_FDR05.csv")
# show
print("df_cat size:", df_cat.shape)
df_cat.head(3)

df_cat size: (28, 5)


,Term,NES,FDR q-val,mlog10_fdr,Project
0,Oxidative Phosphorylation,-1.297360,0.423275,0.373378,PRJNA774885
1,Fatty Acid Metabolism,-0.949103,0.680646,0.167079,PRJNA774885
2,Myc Targets V1,-0.986581,0.632969,0.198618,PRJNA774885
